# MyHeritage low-pass WGS: exploratory population-genetics notebook

**Purpose:** local, reproducible exploration of a MyHeritage CRAM file, including alignment QC, variant-call summaries, population PCA and selected exploratory statistics.

**Important:** MyHeritage describes its WGS CRAM as approximately 2× low-pass sequencing. This notebook does not treat unobserved bases as reference alleles and does not perform clinical interpretation. Genotype calling/imputation is an external, validated bioinformatics stage. Results depend on the reference assembly, sequencing characteristics, caller, imputation panel and sample composition.

## Workflow

1. Validate CRAM, index and reference assembly.
2. Summarize alignment and coverage.
3. Generate a low-pass genotype likelihood/imputation dataset using GLIMPSE2 (outside this notebook).
4. Harmonize your sample with a compatible reference panel (e.g. 1000 Genomes Phase 3 on GRCh37).
5. Compute reference PCA and project the sample.
6. Inspect genotype/variant statistics and sensitivity.

**Do not upload your CRAM, VCF, BCF, PLINK files or sample identifiers to public notebooks or repositories.**

## 0. Project configuration

Use a local Linux/WSL2 environment. Keep the original CRAM read-only. The example paths below are placeholders.

Suggested environment:

```bash
conda create -n dna-explore -c conda-forge -c bioconda \
  python=3.11 pandas numpy scipy matplotlib seaborn scikit-learn \
  pysam cyvcf2 samtools bcftools plink2 jupyterlab
conda activate dna-explore
```

Install GLIMPSE2 separately following its official documentation. Pin exact tool versions after a successful test run; save `conda list --explicit` and tool version outputs in your project.

Suggested layout:

```text
project/
  data/private/sample.cram
  data/private/sample.cram.crai
  data/reference/
  data/intermediate/
  results/
  notebooks/
```

Add private and intermediate data directories to `.gitignore`.

In [ ]:
from pathlib import Path
import os, sys, platform, subprocess, json, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path.cwd().resolve()
CRAM = PROJECT / "data/private/sample.cram"
CRAI = Path(str(CRAM) + ".crai")
REFERENCE_FASTA = PROJECT / "data/reference/reference.fa"
OUT = PROJECT / "results"
INTERMEDIATE = PROJECT / "data/intermediate"

for p in (OUT, INTERMEDIATE):
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 2026
sns.set_theme(context="notebook", style="whitegrid")

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Project:", PROJECT)
print("CRAM exists:", CRAM.exists())
print("Index exists:", CRAI.exists())
print("Reference FASTA exists:", REFERENCE_FASTA.exists())

## 1. Validate the CRAM and identify its reference assembly

CRAM decoding requires the correct reference sequence. Do not infer GRCh37/GRCh38 from a filename. Confirm the assembly from MyHeritage documentation/support or the file's provenance, then obtain the exact FASTA and index it.

In [ ]:
import pysam

assert CRAM.exists(), f"CRAM not found: {CRAM}"
assert REFERENCE_FASTA.exists(), f"Reference FASTA not found: {REFERENCE_FASTA}"

# Check CRAM integrity (requires samtools executable).
subprocess.run(["samtools", "quickcheck", "-v", str(CRAM)], check=True)

# Inspect header without printing potentially identifying sample/read-group fields.
with pysam.AlignmentFile(str(CRAM), "rc", reference_filename=str(REFERENCE_FASTA)) as bam:
    header = bam.header.to_dict()
    sq = header.get("SQ", [])
    print("Number of reference sequences:", len(sq))
    print("First reference sequences:")
    for item in sq[:12]:
        print({k: item.get(k) for k in ("SN", "LN", "AS", "M5") if k in item})
    print("Program records:", [
        {k: v for k, v in x.items() if k not in ("CL",)}
        for x in header.get("PG", [])
    ])

# Build the FASTA index if absent.
if not Path(str(REFERENCE_FASTA) + ".fai").exists():
    subprocess.run(["samtools", "faidx", str(REFERENCE_FASTA)], check=True)

# Create CRAI if absent.
if not CRAI.exists():
    subprocess.run(["samtools", "index", str(CRAM)], check=True)

print("CRAM validation completed.")

### External command-line QC

Run these in a terminal (not as Python statements):

```bash
samtools flagstat data/private/sample.cram > results/flagstat.txt
samtools idxstats data/private/sample.cram > results/idxstats.tsv
samtools stats data/private/sample.cram > results/samtools_stats.txt
```

`idxstats` requires an index. The reference sequence dictionary and CRAM decoding must be compatible. Keep the full header private because read-group metadata may identify the sample or laboratory.

In [ ]:
# Parse idxstats output and plot mapped/unmapped read counts by reference sequence.
idxstats_file = OUT / "idxstats.tsv"
if idxstats_file.exists():
    idx = pd.read_csv(
        idxstats_file, sep="\t", header=None,
        names=["sequence", "length", "mapped", "unmapped"]
    )
    display(idx.head())
    plot_idx = idx[~idx["sequence"].isin(["*", "MT", "chrM"])].copy()
    plot_idx = plot_idx.sort_values("sequence")
    ax = plot_idx.set_index("sequence")[["mapped", "unmapped"]].plot(
        kind="bar", stacked=True, figsize=(14, 4)
    )
    ax.set_ylabel("Read segments")
    ax.set_xlabel("Reference sequence")
    plt.tight_layout()
    plt.show()
else:
    print("Run samtools idxstats first; expected:", idxstats_file)

## 2. Coverage assessment

At low-pass depth, mean coverage alone is insufficient. Inspect the distribution of per-base depth, including zero coverage and high-depth tails. `samtools coverage` provides a convenient per-reference summary. For a more detailed distribution, use `samtools depth`; it can produce a large file, so consider autosomes or selected regions first.

```bash
samtools coverage data/private/sample.cram > results/coverage.tsv
# Potentially large output; restrict to autosomes if needed:
samtools depth -aa -r chr1 data/private/sample.cram > results/chr1.depth.tsv
```

Chromosome naming may be `1` rather than `chr1`; inspect the CRAM header first.

In [ ]:
coverage_file = OUT / "coverage.tsv"
if coverage_file.exists():
    cov = pd.read_csv(coverage_file, sep="\t", comment="#")
    display(cov.head())
    # samtools coverage column names vary slightly by version; inspect before plotting.
    print("Coverage columns:", cov.columns.tolist())
else:
    print("Run samtools coverage first; expected:", coverage_file)

In [ ]:
depth_file = OUT / "chr1.depth.tsv"
if depth_file.exists():
    depth = pd.read_csv(
        depth_file, sep="\t", header=None,
        names=["chromosome", "position", "depth"]
    )
    display(depth["depth"].describe(percentiles=[.1, .25, .5, .75, .9, .99]))
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.histplot(depth["depth"].clip(upper=15), bins=np.arange(0, 17)-0.5, ax=ax)
    ax.set(xlabel="Read depth (values >15 grouped at 15)", ylabel="Bases",
           title="Chromosome 1 depth distribution")
    plt.tight_layout()
    plt.show()
else:
    print("Generate chr1.depth.tsv with samtools depth before this cell.")

## 3. Low-pass genotype calling and imputation (external stage)

**Do not use naive hard calls from a 2× CRAM as the main genotype matrix.** Low-pass data have many zero-read sites and uncertain heterozygous calls. Use genotype likelihoods and a validated reference-based imputation workflow.

GLIMPSE2 is designed for low-coverage WGS and produces GT, dosage (DS), genotype probabilities (GP) and imputation information. Follow its official tutorial for:
- compatible FASTA and genetic map;
- compatible phased reference panel;
- chunk definition and reference splitting;
- CRAM input and sample list;
- per-chunk phasing/imputation and ligation.

Official docs: https://odelaneau.github.io/GLIMPSE/

**Critical compatibility issue:** 1000 Genomes Phase 3 is GRCh37. If the CRAM uses GRCh38, either choose a compatible GRCh38 reference/imputation resource or perform a carefully validated liftover of variants and reference resources. Do not mix assemblies.

Example outputs expected for the downstream notebook:
- `data/intermediate/sample.imputed.vcf.gz` (or BCF), with DS/GP/INFO fields;
- a matched reference panel genotype dataset;
- a sample metadata panel.

This notebook intentionally does not provide a one-line GLIMPSE2 command: its commands depend on the exact reference panel, map, assembly, contig naming, chunk files and GLIMPSE2 version. Copy the complete command sequence from the matching GLIMPSE2 tutorial and record all parameters.

In [ ]:
# Inspect an imputed VCF/BCF after GLIMPSE2.
from pathlib import Path
IMPUTED_VCF = INTERMEDIATE / "sample.imputed.vcf.gz"

if IMPUTED_VCF.exists():
    subprocess.run(["bcftools", "index", "-t", str(IMPUTED_VCF)], check=False)
    print(subprocess.run(
        ["bcftools", "query", "-f",
         "%CHROM\t%POS\t%ID\t%REF\t%ALT[\t%GT\t%DS\t%GP]\n",
         str(IMPUTED_VCF)],
        capture_output=True, text=True, check=True
    ).stdout.splitlines()[:5])
else:
    print("Imputed VCF not present yet:", IMPUTED_VCF)

## 4. Variant-level QC and descriptive statistics

The following code uses `cyvcf2` to inspect an imputed VCF. It summarizes variant counts and, when available, INFO and dosage. These are exploratory summaries, not a clinical variant report.

For population analysis, restrict to high-quality, common, biallelic autosomal SNPs, harmonize alleles and apply LD pruning. Rare variants and indels require separate handling.

In [ ]:
from cyvcf2 import VCF

def summarize_vcf(vcf_path):
    vcf = VCF(str(vcf_path))
    rows = []
    for v in vcf:
        if len(v.ALT) != 1:
            continue
        if len(v.REF) != 1 or len(v.ALT[0]) != 1:
            continue
        if v.CHROM not in {str(i) for i in range(1, 23)} and v.CHROM not in {f"chr{i}" for i in range(1, 23)}:
            continue
        info_score = v.INFO.get("INFO")
        rows.append({
            "chrom": v.CHROM, "pos": v.POS, "id": v.ID,
            "ref": v.REF, "alt": v.ALT[0], "info": info_score
        })
    vcf.close()
    return pd.DataFrame(rows)

if IMPUTED_VCF.exists():
    variants = summarize_vcf(IMPUTED_VCF)
    print("Biallelic autosomal SNPs:", len(variants))
    display(variants.head())
    if variants["info"].notna().any():
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(variants["info"].dropna(), bins=30, ax=ax)
        ax.set(title="Imputation INFO distribution", xlabel="INFO", ylabel="Variants")
        plt.tight_layout()
        plt.show()
else:
    print("Complete imputation before variant QC.")

## 5. Prepare the reference population panel

A useful starting point is 1000 Genomes Phase 3, whose collection is on GRCh37. It contains diverse global populations, including IBS (Iberian individuals from Spain), GBR, TSI, FIN and CEU. The project portal supplies sample panels and per-chromosome genotype files.

Reference:
https://www.internationalgenome.org/data-portal/data-collection/phase3

Use a matching phased reference panel for GLIMPSE2, and use the same assembly, variant representation and allele coding for PCA. Keep the population panel and sample IDs as supplied. For an exploratory European comparison, include IBS, GBR, TSI, FIN and CEU; retain a global panel for context, but do not interpret continental axes as precise national ancestry.

In [ ]:
PANEL = PROJECT / "data/reference/1000G_phase3.panel"

if PANEL.exists():
    panel = pd.read_csv(PANEL, sep="\t")
    display(panel.head())
    print(panel.columns.tolist())
    print(panel.groupby(panel.columns[1]).size().sort_values(ascending=False).head(20))
else:
    print("Download the official Phase 3 sample panel to:", PANEL)

## 6. PCA: reference fit, then sample projection

Avoid fitting PCA jointly to a high-quality reference panel and one low-pass individual with different missingness/uncertainty. Fit PCA on the reference individuals, LD-pruned variants and project the target sample using the reference allele frequencies and loadings.

A robust protocol:
1. Select high-quality, unrelated reference samples.
2. Restrict to autosomal, biallelic SNPs with consistent alleles.
3. Apply missingness and allele-frequency filters.
4. LD-prune the reference SNPs.
5. Fit reference PCA.
6. Project the target using reference loadings/frequencies.
7. Validate projection by holding out reference individuals and projecting them as pseudo-targets.

PLINK 2 docs: https://www.cog-genomics.org/plink/2.0/str

Do not use the illustrative commands from another pipeline without checking the exact PLINK 2 version and output column schema. PLINK projection score scaling must be checked against held-out reference samples.

### Example PLINK 2 reference commands

These commands assume you have already created a harmonized reference PLINK dataset called `ref_qc` and have inspected its sample/variant counts.

```bash
plink2 --pfile data/intermediate/ref_qc \
  --indep-pairwise 200 50 0.2 \
  --out results/ref_prune

plink2 --pfile data/intermediate/ref_qc \
  --extract results/ref_prune.prune.in \
  --freq counts \
  --pca allele-wts 20 \
  --out results/ref_pca
```

The parameters `200 50 0.2` are an illustrative LD-pruning setting, not a universal optimum. Repeat with alternative windows/thresholds and report stability. Use PLINK's current documentation for the precise projection syntax and allele-weight columns.

Expected files for plotting:
- `results/reference_pca.tsv`: IID, PC1, PC2, ...;
- `results/reference_metadata.tsv`: IID, population, superpopulation;
- `results/sample_projection.tsv`: sample label, PC1, PC2, ... on the **same validated scale**.

In [ ]:
REF_PCA = OUT / "reference_pca.tsv"
REF_META = OUT / "reference_metadata.tsv"
SAMPLE_PCS = OUT / "sample_projection.tsv"

required = [REF_PCA, REF_META, SAMPLE_PCS]
if all(p.exists() for p in required):
    ref_pca = pd.read_csv(REF_PCA, sep="\t")
    ref_meta = pd.read_csv(REF_META, sep="\t")
    sample_pcs = pd.read_csv(SAMPLE_PCS, sep="\t")
    print(ref_pca.shape, ref_meta.shape, sample_pcs.shape)
    display(ref_pca.head(), ref_meta.head(), sample_pcs.head())
else:
    print("Create the three harmonized TSV files listed above before plotting.")

In [ ]:
if all(p.exists() for p in required):
    plot_df = ref_pca.merge(ref_meta, on="IID", how="inner")
    european = ["IBS", "GBR", "TSI", "FIN", "CEU"]
    sub = plot_df[plot_df["population"].isin(european)]

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(
        data=sub, x="PC1", y="PC2", hue="population",
        alpha=.65, s=35, ax=ax
    )
    ax.scatter(
        sample_pcs["PC1"], sample_pcs["PC2"],
        marker="*", s=250, edgecolor="black",
        color="black", label="Target sample"
    )
    ax.set_title("European reference PCA with projected sample")
    plt.tight_layout()
    plt.show()
else:
    print("PCA inputs not yet available.")

## 7. Genetic distances and population summaries

A simple descriptive alternative to interpreting only a PCA plot is to calculate distances in the same projected PC space. This is not an ancestry percentage and depends on which PCs, scaling and reference individuals are used.

Use several PCs and report sensitivity to the number of PCs. Prefer population centroids and within-population dispersion rather than the nearest single individual.

In [ ]:
if all(p.exists() for p in required):
    pcs = [c for c in ref_pca.columns if c.startswith("PC")]
    selected_pcs = pcs[:10]
    merged = ref_pca.merge(ref_meta, on="IID", how="inner")

    # Euclidean distances to population centroids in the reference PCA space.
    centroids = merged.groupby("population")[selected_pcs].mean()
    target = sample_pcs[selected_pcs].iloc[0].to_numpy(dtype=float)
    distances = np.sqrt(((centroids - target) ** 2).sum(axis=1))
    distance_table = distances.sort_values().rename("PC_space_distance").to_frame()
    display(distance_table.head(15))
else:
    print("PCA inputs not yet available.")

## 8. Variant annotation (optional)

For selected variants, use Ensembl VEP or another well-documented annotation resource after normalizing and validating variants against the correct assembly. Annotation should include transcript consequences, allele frequencies and evidence sources. Do not interpret an annotation label as a diagnosis.

VEP: https://www.ensembl.org/info/docs/tools/vep/index.html

For ancestry exploration, functional annotation is secondary. Avoid running health-risk interpretation on a consumer low-pass file.

## 9. Statistical robustness and limitations

Minimum sensitivity analyses:
- Compare PCA after multiple LD-pruning thresholds.
- Compare global versus European-only reference PCA.
- Vary imputation INFO thresholds (for example 0.4, 0.6, 0.8), while treating these as sensitivity settings rather than universal cutoffs.
- Compare dosage-based results with high-confidence genotype calls.
- Hold out reference individuals and assess projection accuracy.
- Bootstrap reference individuals within populations to quantify centroid uncertainty.
- Repeat distances using different numbers of PCs.

Interpretation limitations:
- PCA is conditional on the reference panel and variants retained.
- IBS is a sample of Iberian individuals, not a complete representation of Spain or the Basque Country.
- Genetic similarity does not establish nationality, ethnicity, language, cultural identity or a specific ancestor.
- Low-pass imputation quality varies by allele frequency, ancestry match and sequencing characteristics.
- A single individual cannot support reliable population-level inference.
- This is exploratory research, not medical, legal or forensic analysis.

In [ ]:
# Bootstrap population centroids in PCA space (illustrative uncertainty analysis).
# Requires ref_pca/ref_meta/sample_pcs loaded above.
if all(p.exists() for p in required):
    rng = np.random.default_rng(RANDOM_SEED)
    merged = ref_pca.merge(ref_meta, on="IID", how="inner")
    pc_cols = [c for c in ref_pca.columns if c.startswith("PC")][:5]
    target = sample_pcs[pc_cols].iloc[0].to_numpy(float)

    records = []
    for pop, group in merged.groupby("population"):
        if len(group) < 5:
            continue
        boot_d = []
        for _ in range(1000):
            boot = group.sample(
                n=len(group), replace=True,
                random_state=int(rng.integers(0, 2**31-1))
            )
            centroid = boot[pc_cols].mean().to_numpy(float)
            boot_d.append(np.linalg.norm(target - centroid))
        records.append({
            "population": pop,
            "distance_median": np.median(boot_d),
            "distance_low": np.quantile(boot_d, .025),
            "distance_high": np.quantile(boot_d, .975),
            "n_reference": len(group)
        })
    bootstrap_distances = pd.DataFrame(records).sort_values("distance_median")
    display(bootstrap_distances.head(15))
else:
    print("PCA inputs not yet available.")

## 10. Reproducibility and privacy checklist

Before sharing results:
- Record CRAM provenance, genome assembly, FASTA checksum, tool versions, reference-panel release and command-line parameters.
- Keep a copy of the original CRAM unchanged.
- Do not place CRAM/VCF/BCF/PLINK files, sample IDs or raw genotypes in Git or public cloud notebooks.
- Remove identifying metadata from logs and notebook outputs.
- Remember that genetic data can reveal information about biological relatives as well as the tested person.
- Keep intermediate genotype files encrypted and access-controlled.
- Check the current terms and privacy policies before sending data to any third-party service.

MyHeritage states that the downloaded CRAM is the user's responsibility and is no longer protected by the site's security/privacy settings; it also says the file is not intended for medical, health or legal use.

## References
- MyHeritage CRAM download and coverage FAQ: https://www.myheritage.com/help/en/articles/12851869-how-do-i-download-my-raw-dna-data-file-or-cram-file-from-myheritage
- GLIMPSE2 documentation: https://odelaneau.github.io/GLIMPSE/
- 1000 Genomes Phase 3: https://www.internationalgenome.org/data-portal/data-collection/phase3
- PLINK 2: https://www.cog-genomics.org/plink/2.0/
- Ensembl VEP: https://www.ensembl.org/info/docs/tools/vep/index.html

**Notebook status:** This is a reproducible analysis scaffold. It becomes executable end-to-end after you identify the CRAM assembly, obtain the compatible reference resources, run low-pass imputation, and export the documented harmonized PCA input tables. It intentionally does not fabricate genotype calls or sample coordinates.